In [ ]:
!pip install -q streamlit pyngrok pyjwt bcrypt email-validator

In [ ]:
from google.colab import userdata
import os

JWT_SECRET = userdata.get("JWT_SECRET")
NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")
EMAIL_ADDRESS = userdata.get("EMAIL_ADDRESS")
EMAIL_PASSWORD = userdata.get("EMAIL_PASSWORD")

os.environ["JWT_SECRET"] = JWT_SECRET
os.environ["NGROK_AUTHTOKEN"] = NGROK_AUTHTOKEN
os.environ["EMAIL_ADDRESS"] = EMAIL_ADDRESS
os.environ["EMAIL_PASSWORD"] = EMAIL_PASSWORD

print("✅ All secrets loaded successfully!")

In [ ]:
!mkdir -p .streamlit

In [ ]:
%%writefile .streamlit/config.toml
[theme]
base = "light"
primaryColor = "#7c3aed"
backgroundColor = "#f8fafc"
secondaryBackgroundColor = "#ffffff"
textColor = "#1f2937"
font = "sans serif"

In [ ]:
%%writefile db.py

import sqlite3
import bcrypt

DB_NAME = "freight_quote_portal.db"


# ---------------------------
# Database Connection
# ---------------------------
def get_connection():
    return sqlite3.connect(DB_NAME)


# ---------------------------
# Create Tables
# ---------------------------
def init_db():
    conn = get_connection()
    cur = conn.cursor()

    cur.execute("""
    CREATE TABLE IF NOT EXISTS users(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT UNIQUE NOT NULL,
        email TEXT UNIQUE NOT NULL,
        password TEXT NOT NULL,
        security_question TEXT NOT NULL,
        security_answer TEXT NOT NULL,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """)

    conn.commit()
    conn.close()


# ---------------------------
# Password Hashing
# ---------------------------
def hash_password(password):
    return bcrypt.hashpw(password.encode(), bcrypt.gensalt()).decode()


def verify_password(password, hashed):
    return bcrypt.checkpw(password.encode(), hashed.encode())


# ---------------------------
# User Registration
# ---------------------------
def register_user(username, email, password, security_question, security_answer):
    conn = get_connection()
    cur = conn.cursor()
    try:
        cur.execute("""
        INSERT INTO users (username, email, password, security_question, security_answer)
        VALUES (?, ?, ?, ?, ?)
        """, (
            username,
            email,
            hash_password(password),
            security_question,
            hash_password(security_answer)
        ))
        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False
    finally:
        conn.close()


# ---------------------------
# Login (accepts Username OR Email)
# ---------------------------
def get_user_by_identifier(identifier):
    """Returns (username, email, password_hash) matching a username or email, else None."""
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("""
        SELECT username, email, password FROM users
        WHERE username = ? OR email = ?
    """, (identifier, identifier))
    row = cur.fetchone()
    conn.close()
    return row


def authenticate_user(identifier, password):
    row = get_user_by_identifier(identifier)
    if row:
        return verify_password(password, row[2])
    return False


# ---------------------------
# Check Username / Email Exists
# ---------------------------
def username_exists(username):
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("SELECT 1 FROM users WHERE username=?", (username,))
    exists = cur.fetchone()
    conn.close()
    return exists is not None


def email_exists(email):
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("SELECT 1 FROM users WHERE email=?", (email,))
    exists = cur.fetchone()
    conn.close()
    return exists is not None


# ---------------------------
# Security Question / Answer
# ---------------------------
def get_security_question(email):
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("SELECT security_question FROM users WHERE email=?", (email,))
    row = cur.fetchone()
    conn.close()
    return row[0] if row else None


def verify_security_answer(email, answer):
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("SELECT security_answer FROM users WHERE email=?", (email,))
    row = cur.fetchone()
    conn.close()
    if row:
        return verify_password(answer, row[0])
    return False


# ---------------------------
# Update Password
# ---------------------------
def update_password(email, new_password):
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("UPDATE users SET password=? WHERE email=?", (hash_password(new_password), email))
    conn.commit()
    conn.close()


# ---------------------------
# Check New Password vs Old Password
# ---------------------------
def is_same_password(email, new_password):
    """Returns True if new_password matches the user's CURRENT stored password."""
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("SELECT password FROM users WHERE email=?", (email,))
    row = cur.fetchone()
    conn.close()
    if row:
        return verify_password(new_password, row[0])
    return False


# ---------------------------
# Get Username
# ---------------------------
def get_username(email):
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("SELECT username FROM users WHERE email=?", (email,))
    row = cur.fetchone()
    conn.close()
    return row[0] if row else None


# ---------------------------
# Admin: list all registered users (never returns passwords)
# ---------------------------
def get_all_users():
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("SELECT username, email, created_at FROM users ORDER BY created_at DESC")
    rows = cur.fetchall()
    conn.close()
    return rows


def user_count():
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM users")
    n = cur.fetchone()[0]
    conn.close()
    return n


init_db()


In [ ]:
%%writefile otp.py

import random
import smtplib
import os
from datetime import datetime, timedelta
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

# Store OTPs temporarily in memory: email -> {"otp": str, "expires_at": datetime}
otp_storage = {}

OTP_VALIDITY_MINUTES = 5

EMAIL_ADDRESS = os.environ.get("EMAIL_ADDRESS")
EMAIL_PASSWORD = os.environ.get("EMAIL_PASSWORD")


def generate_otp():
    """Generate a 6-digit OTP"""
    return str(random.randint(100000, 999999))


def send_otp(email):
    """
    Generate and send OTP to the given email.
    Returns True if email sent successfully.
    """
    otp = generate_otp()
    otp_storage[email] = {
        "otp": otp,
        "expires_at": datetime.now() + timedelta(minutes=OTP_VALIDITY_MINUTES),
    }

    subject = "Infosys Intelligent Freight Quote - Password Reset OTP"

    text_body = f"""Hello,

Your One-Time Password (OTP) for resetting your password is: {otp}

This OTP will expire in {OTP_VALIDITY_MINUTES} minutes and can be used only once.
If you did not request a password reset, please ignore this email.

Regards,
Infosys Intelligent Freight Quote Authentication Team

© 2026 Infosys Intelligent Freight Quote
"""

    html_body = f"""
    <div style="font-family:'Segoe UI',Arial,sans-serif;max-width:480px;margin:auto;
                border:1px solid #e5e7eb;border-radius:16px;overflow:hidden;
                box-shadow:0 8px 30px rgba(79,70,229,0.12);">
      <div style="background:linear-gradient(135deg,#4f46e5,#7c3aed,#a855f7);padding:30px 24px;text-align:center;">
        <div style="font-size:34px;line-height:1;margin-bottom:8px;">🔐</div>
        <h2 style="color:#ffffff;margin:0;font-size:20px;letter-spacing:-0.01em;">
          Infosys Intelligent Freight Quote
        </h2>
        <p style="color:rgba(255,255,255,0.85);margin:6px 0 0 0;font-size:13px;">
          Password Reset Verification
        </p>
      </div>
      <div style="padding:32px 28px;background:#ffffff;">
        <p style="color:#111827;font-size:15px;margin:0 0 6px 0;">Hello,</p>
        <p style="color:#111827;font-size:15px;margin:0 0 20px 0;">
          Use the One-Time Password (OTP) below to reset your account password.
        </p>
        <div style="text-align:center;margin:24px 0;">
          <span style="display:inline-block;background:#f3f4f6;color:#4f46e5;
                       font-size:30px;font-weight:700;letter-spacing:8px;
                       padding:14px 26px;border-radius:10px;border:1px dashed #c7c2f5;">{otp}</span>
        </div>
        <div style="background:#fff7ed;border:1px solid #fed7aa;border-radius:10px;
                     padding:12px 16px;margin:20px 0;text-align:center;">
          <span style="color:#9a3412;font-size:13px;font-weight:600;">
            ⏱️ This OTP will expire in {OTP_VALIDITY_MINUTES} minutes
          </span>
        </div>
        <p style="color:#6b7280;font-size:13px;margin:18px 0 0 0;">
          This OTP is valid for a single use only. If you did not request a
          password reset, you can safely ignore this email — your account remains secure.
        </p>
      </div>
      <div style="background:#f9fafb;padding:16px;text-align:center;border-top:1px solid #eef0f6;">
        <span style="color:#9ca3af;font-size:12px;">
          © 2026 Infosys Intelligent Freight Quote &middot; Authentication Module
        </span>
      </div>
    </div>
    """

    msg = MIMEMultipart("alternative")
    msg["Subject"] = subject
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = email
    msg.attach(MIMEText(text_body, "plain"))
    msg.attach(MIMEText(html_body, "html"))

    try:
        server = smtplib.SMTP("smtp.gmail.com", 587)
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        server.sendmail(EMAIL_ADDRESS, email, msg.as_string())
        server.quit()
        return True
    except Exception as e:
        print("Email Error:", e)
        return False


def verify_otp(email, otp):
    """
    Verify entered OTP.
    Returns one of: "success", "expired", "invalid"
    """
    record = otp_storage.get(email)

    if not record:
        return "invalid"

    if datetime.now() > record["expires_at"]:
        del otp_storage[email]
        return "expired"

    if record["otp"] == otp:
        del otp_storage[email]
        return "success"

    return "invalid"


def otp_seconds_remaining(email):
    """Returns seconds left before the OTP for this email expires (0 if none/expired)."""
    record = otp_storage.get(email)
    if not record:
        return 0
    remaining = (record["expires_at"] - datetime.now()).total_seconds()
    return max(0, int(remaining))


In [ ]:
%%writefile app.py
import streamlit as st
import re
import jwt
import datetime as dt          # aliased so `from otp import *` below (which
import os                      # brings in `datetime`/`timedelta` classes)
                                # can never shadow the `datetime` module name.
import pandas as pd

from db import *
from otp import *

# =====================================================
# PAGE CONFIG
# =====================================================
st.set_page_config(
    page_title="Infosys Intelligent Freight Quote",
    page_icon="🔐",
    layout="wide",
    initial_sidebar_state="collapsed"
)

JWT_SECRET = os.environ.get("JWT_SECRET", "dev-secret-change-me")

# Hardcoded admin credentials -> defined directly in code, NOT a signup account,
# never stored in the users table.
ADMIN_USERNAME = "admin"
ADMIN_PASSWORD = "Admin@123"

SECURITY_QUESTIONS = [
    "What is your pet name?",
    "What is your favourite city?",
    "What is your mother's maiden name?",
]

EMAIL_PATTERN = r"^[A-Za-z]{2,}@[A-Za-z]{2,}\.[A-Za-z]{2,}$"
USERNAME_PATTERN = r"^[A-Za-z0-9_]{3,20}$"
OTP_PATTERN = r"^[0-9]{6}$"

EMAIL_FORMAT_HELP = "Invalid email format. Please use the format: name@example.com (letters only before/after '@', e.g. john@company.com)."
USERNAME_FORMAT_HELP = "Invalid username. Use 3-20 characters with letters, numbers, and underscores only (e.g. john_doe23)."
PASSWORD_FORMAT_HELP = "Invalid password. It must be at least 8 characters long and include an uppercase letter, a lowercase letter, a number, and a special character (e.g. Test@1234)."
OTP_FORMAT_HELP = "Invalid OTP. Please enter the 6-digit numeric code sent to your email (e.g. 482913)."

# =====================================================
# THEME / CSS
# =====================================================
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700;800&display=swap');

html, body, [class*="css"]  {
    font-family: 'Poppins', sans-serif;
}

#MainMenu, footer, header {visibility: hidden;}

.stApp {
    background:
        radial-gradient(circle at 12% 8%, rgba(236,72,153,0.16) 0%, rgba(236,72,153,0) 42%),
        radial-gradient(circle at 88% 6%, rgba(56,189,248,0.16) 0%, rgba(56,189,248,0) 42%),
        radial-gradient(circle at 15% 92%, rgba(250,204,21,0.14) 0%, rgba(250,204,21,0) 40%),
        radial-gradient(circle at 85% 92%, rgba(124,58,237,0.18) 0%, rgba(124,58,237,0) 45%),
        linear-gradient(160deg, #f5f3ff 0%, #eef2ff 45%, #fdf4ff 100%);
    background-attachment: fixed;
}

@keyframes fadeInUp {
    from { opacity: 0; transform: translateY(14px); }
    to   { opacity: 1; transform: translateY(0); }
}

@keyframes gradientShift {
    0%   { background-position: 0% 50%; }
    50%  { background-position: 100% 50%; }
    100% { background-position: 0% 50%; }
}


/* With layout="wide" we get the full viewport to draw our colorful
   background on, and control card width ourselves via max-width +
   margin:auto below — this replaces the old st.columns((1,2.4,1))
   centering trick, which on narrow/mobile viewports rendered its two
   "empty" spacer columns as their own stacked, blank sections and made
   the whole page feel unnecessarily tall/vertical. */
.block-container {
    padding-top: 2.2rem;
    padding-bottom: 2rem;
    max-width: 100%;
}

/* Center column card — targets the real Streamlit container created via
   st.container(key="auth_card"), so all widgets rendered inside the
   `with` block are genuinely nested in this box (fixes the empty/blank
   card that appeared when the card was built from two separate,
   unlinked st.markdown() div tags). Width is responsive: it fills the
   screen on phones/small tablets and settles to a comfortable fixed
   width on larger screens, instead of a fixed narrow column. */
div.st-key-auth_card,
div.st-key-auth_card_wide {
    position: relative;
    background: #ffffff;
    padding: 2.6rem 2.3rem 2rem 2.3rem;
    border-radius: 22px;
    box-shadow: 0 20px 50px rgba(124, 58, 237, 0.18), 0 2px 8px rgba(0,0,0,0.05);
    border: 1px solid #eef0f6;
    margin: 0.6rem auto 1.2rem auto;
    animation: fadeInUp 0.35s ease-out;
    overflow: hidden;
    width: 100%;
}
div.st-key-auth_card { max-width: 480px; }
div.st-key-auth_card_wide { max-width: 980px; }

div.st-key-auth_card::before,
div.st-key-auth_card_wide::before {
    content: "";
    position: absolute;
    top: 0; left: 0; right: 0;
    height: 6px;
    background: linear-gradient(90deg,#ec4899,#7c3aed,#4f46e5,#38bdf8,#f59e0b);
    background-size: 300% 100%;
    animation: gradientShift 6s ease infinite;
    z-index: 2;
}

@media (max-width: 560px) {
    div.st-key-auth_card,
    div.st-key-auth_card_wide {
        padding: 2rem 1.3rem 1.6rem 1.3rem;
        border-radius: 18px;
    }
    .block-container { padding-left: 0.8rem; padding-right: 0.8rem; }
}

.eyebrow {
    text-align: center;
    text-transform: uppercase;
    letter-spacing: 0.14em;
    font-size: 0.68rem;
    font-weight: 700;
    color: #7c3aed;
    margin-bottom: 0.3rem;
}

.brand-header {
    text-align: center;
    margin-bottom: 0.6rem;
}
.brand-header .icon-badge {
    font-size: 2.1rem;
    background: linear-gradient(135deg,#ec4899,#7c3aed 55%,#4f46e5);
    width: 66px; height: 66px;
    line-height: 66px;
    border-radius: 50%;
    display: inline-block;
    margin-bottom: 0.7rem;
    box-shadow: 0 10px 26px rgba(124,58,237,0.4);
}
.brand-header h1 {
    font-size: 1.55rem;
    font-weight: 800;
    color: #1e1b4b;
    margin: 0;
    letter-spacing: -0.01em;
}
.brand-header p {
    color: #6b7280;
    font-size: 0.92rem;
    margin-top: 0.3rem;
}

/* Buttons: primary = solid gradient CTA, secondary = ghost/outline nav action */
.stButton>button {
    width: 100%;
    border-radius: 10px;
    padding: 0.55rem 1rem;
    font-weight: 600;
    transition: all 0.15s ease-in-out;
}

.stButton>button[kind="primary"] {
    border: none;
    background: linear-gradient(135deg,#ec4899,#7c3aed 55%,#4f46e5);
    background-size: 200% 200%;
    color: white;
}
.stButton>button[kind="primary"]:hover {
    transform: translateY(-1px);
    background-position: 100% 0%;
    box-shadow: 0 10px 24px rgba(124,58,237,0.45);
    color: white;
}

.stButton>button[kind="secondary"] {
    background: #ffffff;
    border: 1.5px solid #e2e4ec;
    color: #4f46e5;
}
.stButton>button[kind="secondary"]:hover {
    border-color: #7c3aed;
    background: #f5f3ff;
    color: #4f46e5;
}
.stButton>button:active { transform: translateY(0px); }

/* Text inputs */
.stTextInput>div>div {
    border-radius: 12px;
    overflow: hidden;
    transition: box-shadow 0.15s ease, transform 0.1s ease;
}
.stTextInput>div>div:has(input:focus) {
    box-shadow: 0 6px 18px rgba(124,58,237,0.18);
}
.stTextInput>div>div>input {
    border-radius: 12px;
    padding: 0.6rem 0.9rem;
    border: 1.5px solid #e0def7;
    border-left: 4px solid #a855f7;
    background: linear-gradient(180deg,#faf9ff 0%,#f5f3ff 100%);
    color: #1f2937 !important;
    -webkit-text-fill-color: #1f2937 !important;
    caret-color: #7c3aed !important;
    font-weight: 500;
    transition: border-color 0.15s ease, background 0.15s ease, box-shadow 0.15s ease;
}
.stTextInput>div>div>input::placeholder {
    color: #9c9bb5 !important;
    -webkit-text-fill-color: #9c9bb5 !important;
    opacity: 1 !important;
    font-weight: 400;
}
.stTextInput>div>div>input:hover {
    border-color: #c7bdf5;
}
.stTextInput>div>div>input:focus {
    border-color: #7c3aed;
    border-left: 4px solid #ec4899;
    box-shadow: 0 0 0 3px rgba(124,58,237,0.16);
    background: #ffffff;
}

/* Selectbox / radio */
div[data-baseweb="select"] > div {
    border-radius: 10px;
    border: 1px solid #e2e4ec !important;
    background: #fbfbfe !important;
    color: #1f2937 !important;
}
div[data-baseweb="select"] span {
    color: #1f2937 !important;
    -webkit-text-fill-color: #1f2937 !important;
}
/* The searchable dropdown has its own hidden text input for typing/filtering.
   Without an explicit background/caret color it can inherit a dark theme
   default, which makes the box look black and hides the typing cursor. */
div[data-baseweb="select"] input {
    color: #1f2937 !important;
    -webkit-text-fill-color: #1f2937 !important;
    caret-color: #1f2937 !important;
    background: transparent !important;
}
div[data-baseweb="popover"] {
    background: #ffffff !important;
}
div[data-baseweb="popover"] ul,
div[data-baseweb="popover"] li {
    background: #ffffff !important;
    color: #1f2937 !important;
}
div[data-baseweb="popover"] li:hover {
    background: #f5f3ff !important;
}
div[data-baseweb="popover"] li,
div[data-baseweb="popover"] span {
    color: #1f2937 !important;
}
div[role="radiogroup"] label,
div[role="radiogroup"] p {
    color: #1f2937 !important;
}

.badge-row { text-align:center; margin-top: 0.4rem; }
.small-link { text-align:center; color:#9ca3af; font-size:0.8rem; margin-top: 0.7rem;}

.pw-meter-bg {
    width: 100%; height: 6px; border-radius: 4px; background: #e5e7eb; margin-top: 4px;
}
.pw-meter-fill { height: 6px; border-radius: 4px; transition: width 0.2s ease; }

.metric-card {
    position: relative;
    background: #ffffff;
    border-radius: 16px;
    padding: 1.2rem 1.3rem;
    border: 1px solid #eef0f6;
    box-shadow: 0 6px 20px rgba(0,0,0,0.05);
    text-align: center;
    overflow: hidden;
    transition: transform 0.15s ease, box-shadow 0.15s ease;
}
.metric-card:hover {
    transform: translateY(-3px);
    box-shadow: 0 14px 30px rgba(0,0,0,0.09);
}
.metric-card::before {
    content: "";
    position: absolute;
    top: 0; left: 0; right: 0;
    height: 4px;
    background: linear-gradient(90deg, var(--mc-a, #4f46e5), var(--mc-b, #7c3aed));
}
.metric-card .icon {
    font-size: 1.4rem; margin-bottom: 0.3rem;
    display: inline-flex; align-items:center; justify-content:center;
    width: 42px; height: 42px; border-radius: 12px;
    background: linear-gradient(135deg, var(--mc-a, #4f46e5), var(--mc-b, #7c3aed));
    box-shadow: 0 8px 18px rgba(0,0,0,0.15);
}
.metric-card .val { font-size:1.7rem; font-weight:800; color:#4f46e5; margin-top:0.4rem; }
.metric-card .lbl { font-size:0.78rem; color:#6b7280; margin-top:0.1rem; text-transform:uppercase; letter-spacing:0.04em; }
.metric-card.mc-pink   { --mc-a:#ec4899; --mc-b:#f472b6; }
.metric-card.mc-blue   { --mc-a:#0ea5e9; --mc-b:#38bdf8; }
.metric-card.mc-green  { --mc-a:#10b981; --mc-b:#34d399; }
.metric-card.mc-amber  { --mc-a:#f59e0b; --mc-b:#fbbf24; }
.metric-card.mc-violet { --mc-a:#7c3aed; --mc-b:#a855f7; }
.metric-card.mc-pink .val   { color:#ec4899 !important; }
.metric-card.mc-blue .val   { color:#0284c7 !important; }
.metric-card.mc-green .val  { color:#059669 !important; }
.metric-card.mc-amber .val  { color:#d97706 !important; }
.metric-card.mc-violet .val { color:#7c3aed !important; }


/* Dashboard detail rows */
.detail-row {
    display: flex;
    align-items: center;
    gap: 0.8rem;
    background: #f9fafb;
    border: 1px solid #eef0f6;
    border-radius: 12px;
    padding: 0.75rem 1rem;
    margin-bottom: 0.6rem;
}
.detail-row .d-icon {
    font-size: 1.15rem;
    background: linear-gradient(135deg,#4f46e5,#7c3aed);
    color: white;
    width: 34px; height: 34px;
    border-radius: 9px;
    display: flex; align-items:center; justify-content:center;
    flex-shrink: 0;
}
.detail-row .d-text .d-label { font-size: 0.72rem; color:#9ca3af; text-transform:uppercase; letter-spacing:0.05em; }
.detail-row .d-text .d-value { font-size: 0.95rem; color:#1f2937; font-weight:600; }

/* Force readable text colors on the specific elements that need it,
   without a blanket rule that would also override button/badge text colors */
.stApp, p, span, div, label {
    color: #1f2937;
}
h1, h2, h3, h4, h5, h6 { color: #1e1b4b !important; }

/* Any text inside a button must always inherit the button's own color
   (white on primary, purple on secondary) instead of the page default */
.stButton>button, .stButton>button * {
    color: inherit !important;
}
.stButton>button[kind="primary"], .stButton>button[kind="primary"] * {
    color: #ffffff !important;
}
.stButton>button[kind="secondary"], .stButton>button[kind="secondary"] * {
    color: #4f46e5 !important;
}

/* Same protection for our custom colored badges/labels so the blanket
   rule above never dims them */
.eyebrow, .eyebrow * { color: #7c3aed !important; }
.metric-card .val, .metric-card .val * { color: #4f46e5; }
.metric-card.mc-pink .val, .metric-card.mc-pink .val * { color: #ec4899 !important; }
.metric-card.mc-blue .val, .metric-card.mc-blue .val * { color: #0284c7 !important; }
.metric-card.mc-green .val, .metric-card.mc-green .val * { color: #059669 !important; }
.metric-card.mc-amber .val, .metric-card.mc-amber .val * { color: #d97706 !important; }
.metric-card.mc-violet .val, .metric-card.mc-violet .val * { color: #7c3aed !important; }
.metric-card .lbl, .metric-card .lbl * { color: #6b7280 !important; }
.detail-row .d-icon, .detail-row .d-icon * { color: #ffffff !important; }
.detail-row .d-label, .detail-row .d-label * { color: #9ca3af !important; }
.detail-row .d-value, .detail-row .d-value * { color: #1f2937 !important; }
.brand-header p, .brand-header p * { color: #6b7280 !important; }
.small-link, .small-link * { color: #9ca3af !important; }

div[data-testid="stWidgetLabel"] label,
div[data-testid="stWidgetLabel"] p {
    color: #374151 !important;
    font-weight: 500 !important;
}

.stCaption, div[data-testid="stCaptionContainer"] p {
    color: #6b7280 !important;
}

div[data-testid="stMarkdownContainer"] p {
    color: #1f2937 !important;
}

div.st-key-auth_card hr,
div.st-key-auth_card_wide hr {
    border: none;
    height: 2px;
    background: linear-gradient(90deg, rgba(236,72,153,0.35), rgba(124,58,237,0.35), rgba(56,189,248,0.35));
    border-radius: 2px;
}

table, .stTable, div[data-testid="stTable"] {
    color: #1f2937 !important;
}
thead tr th, tbody tr td {
    color: #1f2937 !important;
}

div[data-testid="stDataFrame"] {
    border-radius: 12px;
    overflow: hidden;
    border: 1px solid #eef0f6;
}
</style>
""", unsafe_allow_html=True)


def brand_header(icon, title, subtitle, eyebrow=None):
    eyebrow_html = f'<div class="eyebrow">{eyebrow}</div>' if eyebrow else ""
    st.markdown(f"""
    {eyebrow_html}
    <div class="brand-header">
        <div class="icon-badge">{icon}</div>
        <h1>{title}</h1>
        <p>{subtitle}</p>
    </div>
    """, unsafe_allow_html=True)


def detail_row(icon, label, value):
    st.markdown(f"""
    <div class="detail-row">
        <div class="d-icon">{icon}</div>
        <div class="d-text">
            <div class="d-label">{label}</div>
            <div class="d-value">{value}</div>
        </div>
    </div>
    """, unsafe_allow_html=True)


def password_strength(password):
    """Returns (label, pct, color) for a simple visual strength meter."""
    if not password:
        return "", 0, "#e5e7eb"
    score = 0
    if len(password) >= 8:
        score += 1
    if re.search("[A-Z]", password):
        score += 1
    if re.search("[a-z]", password):
        score += 1
    if re.search("[0-9]", password):
        score += 1
    if re.search("[^A-Za-z0-9]", password):
        score += 1

    levels = {
        0: ("Very weak", 10, "#ef4444"),
        1: ("Weak", 25, "#ef4444"),
        2: ("Fair", 50, "#f59e0b"),
        3: ("Good", 70, "#f59e0b"),
        4: ("Strong", 90, "#10b981"),
        5: ("Excellent", 100, "#10b981"),
    }
    return levels[score]


def render_password_meter(password):
    label, pct, color = password_strength(password)
    if password:
        st.markdown(f"""
        <div class="pw-meter-bg">
            <div class="pw-meter-fill" style="width:{pct}%; background:{color};"></div>
        </div>
        <div style="font-size:0.78rem; color:{color}; margin-top:2px;">{label}</div>
        """, unsafe_allow_html=True)


# =====================================================
# SESSION STATE
# =====================================================
defaults = {
    "page": "Login",
    "token": None,
    "reset_email": "",
    "otp_sent": False,
    "verified": False,
    "is_admin": False,
    "sq_verified": False,
    "sq_reset_email": "",
}
for k, v in defaults.items():
    if k not in st.session_state:
        st.session_state[k] = v


# =====================================================
# JWT HELPERS
# =====================================================
def create_token(payload_extra):
    payload = {
        **payload_extra,
        "exp": dt.datetime.utcnow() + dt.timedelta(hours=2),
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")


def verify_token(token):
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except Exception:
        return None


def valid_email(email):
    return re.match(EMAIL_PATTERN, email)


def valid_username(username):
    return re.match(USERNAME_PATTERN, username)


def valid_otp_format(otp):
    return re.match(OTP_PATTERN, otp)


def valid_password(password):
    if len(password) < 8:
        return False
    if not re.search("[A-Z]", password):
        return False
    if not re.search("[a-z]", password):
        return False
    if not re.search("[0-9]", password):
        return False
    if not re.search("[^A-Za-z0-9]", password):
        return False
    return True


def goto(page):
    st.session_state.page = page
    st.rerun()


# =====================================================
# LOGIN
# =====================================================
if st.session_state.page == "Login":
    with st.container(key="auth_card"):
        brand_header("🔐", "Welcome Back", "Sign in to your Infosys Intelligent Freight Quote account", eyebrow="Secure Access")

        identifier = st.text_input("👤 Username or Email", placeholder="yourname or you@example.com")
        password = st.text_input("🔒 Password", type="password", placeholder="••••••••")

        if st.button("Login", type="primary"):
            if identifier == "":
                st.error("Username or Email is required.")
            elif password == "":
                st.error("Password is required.")
            elif authenticate_user(identifier, password):
                row = get_user_by_identifier(identifier)
                email = row[1]
                st.session_state.token = create_token({"email": email})
                st.session_state.is_admin = False
                goto("Dashboard")
            else:
                st.error("Invalid username/email or password.")

        st.markdown("<hr style='margin:1.1rem 0 0.6rem 0;'>", unsafe_allow_html=True)

        c1, c2 = st.columns(2)
        with c1:
            if st.button("Create Account", type="secondary"):
                goto("Signup")
        with c2:
            if st.button("Forgot Password", type="secondary"):
                goto("Forgot")

        st.markdown(
            "<div class='small-link'>Portal administrator?</div>",
            unsafe_allow_html=True,
        )
        if st.button("Admin Login", type="secondary", key="go_admin"):
            goto("AdminLogin")


# =====================================================
# SIGNUP
# =====================================================
elif st.session_state.page == "Signup":
    with st.container(key="auth_card"):
        brand_header("📝", "Create Account", "Join the Infosys Intelligent Freight Quote in a minute", eyebrow="Get Started")

        username = st.text_input("👤 Username")
        email = st.text_input("✉️ Email")
        password = st.text_input("🔒 Password", type="password")
        render_password_meter(password)
        confirm = st.text_input("🔒 Confirm Password", type="password")
        question = st.selectbox("🛡️ Security Question", SECURITY_QUESTIONS)
        answer = st.text_input("💬 Security Answer")

        st.caption("Username must be 3-20 characters (letters, numbers, underscores only).")
        st.caption("Password must have 8+ characters, an uppercase & lowercase letter, a number and a special symbol.")

        if st.button("Create Account", type="primary", key="signup_btn"):
            if username == "":
                st.error("Username is required.")
            elif email == "":
                st.error("Email is required.")
            elif password == "":
                st.error("Password is required.")
            elif confirm == "":
                st.error("Confirm Password is required.")
            elif answer == "":
                st.error("Security Answer is required.")
            elif not valid_username(username):
                st.error(USERNAME_FORMAT_HELP)
            elif not valid_email(email):
                st.error(EMAIL_FORMAT_HELP)
            elif not valid_password(password):
                st.error(PASSWORD_FORMAT_HELP)
            elif password != confirm:
                st.error("Passwords do not match. Please re-enter the same password in both fields.")
            elif username_exists(username):
                st.error("Username already exists. Please choose a different username.")
            elif email_exists(email):
                st.error("Email already exists. Please login or use a different email.")
            else:
                ok = register_user(username, email, password, question, answer)
                if ok:
                    st.success("Registration successful! Please login.")
                    goto("Login")
                else:
                    st.error("Registration failed. Please try again.")

        if st.button("Back to Login", type="secondary", key="signup_back"):
            goto("Login")

# =====================================================
# FORGOT PASSWORD
# =====================================================
elif st.session_state.page == "Forgot":
    with st.container(key="auth_card"):
        brand_header("🔑", "Forgot Password", "Recover access using your security question or OTP", eyebrow="Account Recovery")

        method = st.radio("Choose Recovery Method", ["Security Question", "OTP"], horizontal=True)

        # --------------------------
        # Security Question Method
        # --------------------------
        if method == "Security Question":
            email = st.text_input("✉️ Registered Email", key="sq_email")

            if email:
                if email_exists(email):
                    question = get_security_question(email)
                    st.info(f"❓ {question}")

                    already_verified = (
                        st.session_state.sq_verified
                        and st.session_state.sq_reset_email == email
                    )

                    if not already_verified:
                        answer = st.text_input(
                            "💬 Answer",
                            key="sq_answer",
                            help="Your answer is case-sensitive — enter it exactly as you did during signup.",
                        )

                        if st.button("Verify Answer", type="primary", key="sq_verify_btn"):
                            if answer == "":
                                st.error("Security Answer is required.")
                            elif verify_security_answer(email, answer):
                                st.session_state.sq_verified = True
                                st.session_state.sq_reset_email = email
                                st.success("✅ Security answer verified successfully.")
                                st.rerun()
                            else:
                                st.error("❌ Incorrect security answer. Please check your answer (it is case-sensitive) and try again.")
                    else:
                        st.success("✅ Security answer verified. You can now set a new password.")

                        new_password = st.text_input("🔒 New Password", type="password", key="sq_pw")
                        render_password_meter(new_password)
                        confirm = st.text_input("🔒 Confirm Password", type="password", key="sq_cp")

                        if st.button("Reset Password", type="primary", key="sq_reset"):
                            if new_password == "":
                                st.error("Password is required.")
                            elif confirm == "":
                                st.error("Confirm Password is required.")
                            elif not valid_password(new_password):
                                st.error(PASSWORD_FORMAT_HELP)
                            elif new_password != confirm:
                                st.error("Passwords do not match. Please re-enter the same password in both fields.")
                            elif is_same_password(email, new_password):
                                st.error("⚠️ New password cannot be the same as your old password. Please choose a different password.")
                            else:
                                update_password(email, new_password)
                                st.success("🎉 Password updated successfully.")
                                st.session_state.sq_verified = False
                                st.session_state.sq_reset_email = ""
                                goto("Login")
                else:
                    st.error("Email not registered.")

        # --------------------------
        # OTP Method
        # --------------------------
        else:
            email = st.text_input("✉️ Registered Email", key="otp_email")

            if not st.session_state.otp_sent:
                if st.button("Send OTP", type="primary"):
                    if email == "":
                        st.error("Email is required.")
                    elif not valid_email(email):
                        st.error(EMAIL_FORMAT_HELP)
                    elif not email_exists(email):
                        st.error("Email not registered.")
                    elif send_otp(email):
                        st.session_state.otp_sent = True
                        st.session_state.reset_email = email
                        st.rerun()
                    else:
                        st.error("Unable to send OTP. Please try again.")
            else:
                st.success(f"✅ OTP sent successfully to {st.session_state.reset_email}. It will expire in 5 minutes.")

                otp = st.text_input("🔢 Enter OTP", key="otp_input")

                bcol1, bcol2 = st.columns(2)
                with bcol1:
                    verify_clicked = st.button("Verify OTP", type="primary", key="otp_verify_btn")
                with bcol2:
                    resend_clicked = st.button("Resend OTP", type="secondary", key="otp_resend_btn")

                if resend_clicked:
                    if send_otp(st.session_state.reset_email):
                        st.session_state.verified = False
                        st.success("✅ A new OTP has been sent to your email. It will expire in 5 minutes.")
                    else:
                        st.error("Unable to resend OTP. Please try again.")

                if verify_clicked:
                    if otp == "":
                        st.error("OTP is required.")
                    elif not valid_otp_format(otp):
                        st.error(OTP_FORMAT_HELP)
                    else:
                        result = verify_otp(st.session_state.reset_email, otp)
                        if result == "success":
                            st.session_state.verified = True
                            st.rerun()
                        elif result == "expired":
                            st.error("⏱️ Your OTP has expired. Please click 'Resend OTP' to get a new code.")
                        else:
                            st.error("❌ Invalid OTP. Please check the code sent to your email and try again.")

                if st.session_state.verified:
                    st.success("✅ OTP verified. You can now set a new password.")
                    new_password = st.text_input("🔒 New Password", type="password", key="otp_pw")
                    render_password_meter(new_password)
                    confirm = st.text_input("🔒 Confirm Password", type="password", key="otp_cp")

                    if st.button("Update Password", type="primary", key="otp_update_btn"):
                        if new_password == "":
                            st.error("Password is required.")
                        elif confirm == "":
                            st.error("Confirm Password is required.")
                        elif not valid_password(new_password):
                            st.error(PASSWORD_FORMAT_HELP)
                        elif new_password != confirm:
                            st.error("Passwords do not match. Please re-enter the same password in both fields.")
                        elif is_same_password(st.session_state.reset_email, new_password):
                            st.error("⚠️ New password cannot be the same as your old password. Please choose a different password.")
                        else:
                            update_password(st.session_state.reset_email, new_password)
                            st.success("🎉 Password updated successfully.")
                            st.session_state.otp_sent = False
                            st.session_state.verified = False
                            goto("Login")

        st.markdown("<hr style='margin:1.1rem 0 0.6rem 0;'>", unsafe_allow_html=True)
        if st.button("Back to Login", type="secondary", key="forgot_back"):
            st.session_state.otp_sent = False
            st.session_state.verified = False
            st.session_state.sq_verified = False
            st.session_state.sq_reset_email = ""
            goto("Login")

# =====================================================
# ADMIN LOGIN  (separate page, hardcoded credentials, no signup account)
# =====================================================
elif st.session_state.page == "AdminLogin":
    with st.container(key="auth_card"):
        brand_header("🛡️", "Admin Login", "Restricted access for portal administrators", eyebrow="Admin Only")

        admin_user = st.text_input("👤 Admin Username")
        admin_pass = st.text_input("🔒 Admin Password", type="password")

        if st.button("Login as Admin", type="primary"):
            if admin_user == "":
                st.error("Admin Username is required.")
            elif admin_pass == "":
                st.error("Admin Password is required.")
            elif admin_user == ADMIN_USERNAME and admin_pass == ADMIN_PASSWORD:
                st.session_state.token = create_token({"role": "admin", "username": ADMIN_USERNAME})
                st.session_state.is_admin = True
                goto("Dashboard")
            else:
                st.error("Invalid admin credentials.")

        st.markdown("<hr style='margin:1.1rem 0 0.6rem 0;'>", unsafe_allow_html=True)
        if st.button("Back to Login", type="secondary", key="admin_back"):
            goto("Login")

# =====================================================
# DASHBOARD (User or Admin)
# =====================================================
elif st.session_state.page == "Dashboard":
    payload = verify_token(st.session_state.token)

    if payload is None:
        st.error("Session expired. Please login again.")
        st.session_state.token = None
        goto("Login")

    # -----------------------------
    # ADMIN DASHBOARD
    # -----------------------------
    if payload.get("role") == "admin":
        with st.container(key="auth_card_wide"):
            brand_header("🛡️", "Admin Dashboard", f"Signed in as {payload.get('username')}", eyebrow="Administrator")

            users = get_all_users()
            total = user_count()

            # Parse "created_at" safely regardless of exact SQLite timestamp
            # formatting, so a stray format quirk can never crash the page.
            joined_series = pd.to_datetime(
                [u[2] for u in users], errors="coerce"
            ) if users else pd.to_datetime([])

            now = dt.datetime.now()
            today_count = int((joined_series.date == now.date()).sum()) if len(joined_series) else 0
            week_cutoff = now - dt.timedelta(days=7)
            week_count = int((joined_series >= week_cutoff).sum()) if len(joined_series) else 0

            m1, m2, m3, m4 = st.columns(4)
            with m1:
                st.markdown(f"""<div class="metric-card mc-violet"><div class="icon">👥</div><div class="val">{total}</div>
                            <div class="lbl">Total Users</div></div>""", unsafe_allow_html=True)
            with m2:
                st.markdown(f"""<div class="metric-card mc-pink"><div class="icon">✨</div><div class="val">{today_count}</div>
                            <div class="lbl">New Today</div></div>""", unsafe_allow_html=True)
            with m3:
                st.markdown(f"""<div class="metric-card mc-blue"><div class="icon">📈</div><div class="val">{week_count}</div>
                            <div class="lbl">New This Week</div></div>""", unsafe_allow_html=True)
            with m4:
                st.markdown("""<div class="metric-card mc-green"><div class="icon">🟢</div><div class="val">Online</div>
                            <div class="lbl">Portal Status</div></div>""", unsafe_allow_html=True)

            st.write("")

            # ---------------------------------------------
            # Signup activity chart (last 14 days)
            # ---------------------------------------------
            if len(joined_series) and joined_series.notna().any():
                st.markdown("##### 📊 Signup Activity (last 14 days)")
                day_index = pd.date_range(end=now.date(), periods=14, freq="D")
                daily_counts = (
                    pd.Series(1, index=joined_series.dropna().normalize())
                    .groupby(level=0).sum()
                    .reindex(day_index, fill_value=0)
                )
                daily_counts.index = daily_counts.index.strftime("%b %d")
                st.bar_chart(daily_counts, color="#7c3aed", use_container_width=True)
                st.write("")

            st.subheader("Registered Users")
            if len(users) == 0:
                st.info("No registered users yet.")
            else:
                search = st.text_input("🔍 Search by username or email", key="admin_search", placeholder="Type to filter...")
                data = [{"Username": u[0], "Email": u[1], "Joined": u[2]} for u in users]
                if search:
                    s = search.lower()
                    data = [d for d in data if s in d["Username"].lower() or s in d["Email"].lower()]
                st.dataframe(data, use_container_width=True, hide_index=True)
                st.caption(f"Showing {len(data)} of {total} registered user(s).")

            st.markdown("<hr>", unsafe_allow_html=True)
            if st.button("Logout", type="secondary", key="admin_logout"):
                st.session_state.token = None
                st.session_state.is_admin = False
                goto("Login")

    # -----------------------------
    # NORMAL USER DASHBOARD
    # -----------------------------
    else:
        email = payload["email"]
        username = get_username(email)

        with st.container(key="auth_card"):
            brand_header("🏠", f"Welcome, {username}!", "You're securely logged in", eyebrow="User Dashboard")

            detail_row("👤", "Username", username)
            detail_row("✉️", "Email", email)

            st.markdown("<hr>", unsafe_allow_html=True)
            st.info("You have full access to your account. Stay secure!")

            if st.button("Logout", type="secondary"):
                st.session_state.token = None
                goto("Login")


In [ ]:
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokHTTPError
import subprocess
import time
import os

# Authenticate ngrok
ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])

# --- Clean up anything left over from a previous run ---
# 1. Kill any lingering Streamlit process
!pkill -f streamlit

# 2. Disconnect any tunnels pyngrok still thinks are open in THIS session
try:
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
except Exception as e:
    print("No existing tunnels to disconnect:", e)

# 3. Kill the ngrok agent process itself (this frees up the free-tier
#    static domain *for THIS machine* — pkill on streamlit alone does NOT do this)
ngrok.kill()

time.sleep(2)

# Start Streamlit
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"]
)

# Wait for Streamlit to start
time.sleep(8)


def is_endpoint_conflict(err):
    msg = str(err)
    return "ERR_NGROK_334" in msg or "already online" in msg


# --- Create public URL ---
# Free ngrok accounts get ONE fixed domain that's tied to your auth token
# (not randomly generated per run). Occasionally ngrok's cloud takes a few
# extra seconds to notice a previous agent session has disconnected, so we
# retry a couple of times with a short delay before giving up.
public_url = None
last_error = None
for attempt in range(3):
    try:
        public_url = ngrok.connect(8501)
        break
    except PyngrokNgrokHTTPError as e:
        last_error = e
        if is_endpoint_conflict(e):
            print(f"⚠️ Attempt {attempt + 1}/3: domain still shows as online elsewhere, retrying in 6s...")
            time.sleep(6)
        else:
            raise

if public_url:
    print("🌐 Open your app here:")
    print(public_url.public_url)
else:
    # Pooling isn't a fix here: ngrok only load-balances between agents that
    # BOTH started with --pooling-enabled, and we don't control the other,
    # already-running session — so retrying with pooling on our side alone
    # fails the same way. The domain is genuinely held by another live
    # ngrok agent session (a different Colab runtime/tab, most likely),
    # and only stopping THAT session actually frees it up. Do one of:
    #
    #   1. Open https://dashboard.ngrok.com/tunnels/agents, find the active
    #      agent session there, and click to stop/disconnect it — then
    #      just re-run this cell.
    #   2. If you have another Colab tab/notebook running this project,
    #      go to its Runtime menu -> "Disconnect and delete runtime"
    #      (not just "Restart"), which fully kills its ngrok agent too.
    print("❌ Could not start the tunnel — the domain is held by another live ngrok session.")
    print("   Fix: open https://dashboard.ngrok.com/tunnels/agents, stop the active agent")
    print("   session shown there (or disconnect any other Colab runtime using this project),")
    print("   then re-run this cell.")
    raise last_error
